# Neural Networks

FIN 43800 · Machine Learning in Finance

Andreas Neuhierl

## Outline

- What networks do, and a little history
- One hidden layer
- How a network learns: backpropagation by hand, the dials
- Networks on real data: the homes, then a finance application, Gu,
  Kelly & Xiu (2020)
- Networks with memory: recurrent networks
- What networks can do that forests and boosted trees cannot

## Upload, Then Run Setup

Starting a new Colab session? Colab deletes uploaded files when a
session ends. Download `ames_housing.csv`, `size_value_monthly.csv` and
`gw_monthly.csv` from Brightspace, drag them into the Files pane (folder
icon on the left), then run the two **Setup** cells below before any
other cell.

## Setup: What It Builds

- **`homes`** and **`log_price`**: 1,460 home sales in Ames, Iowa,
  2006–2010: 79 features each, and the price in logs.
- **`french`**: Kenneth French’s monthly portfolios and factors,
  1963–2025, for one beta.
- **`monthly`**: the Goyal–Welch file: the stock market’s monthly return
  over T-bills, from 1926.

<!-- -->

- **Your Turn** code is text in this notebook: click **+ Code**, paste
  it, fill the blank, then Shift + Enter.

Run this first, every session.

In [ ]:
import sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Fetch the one helper file we need from the course repo.
    import subprocess
    subprocess.run(
        ["wget", "-q", "-O", "mlfin_data.py",
         "https://raw.githubusercontent.com/aneuhierl/ml-in-finance-fall2026/main/src/mlfin_data.py"],
        check=True,
    )
else:
    # Find src/mlfin_data.py by walking UP from this notebook's folder.
    here = Path.cwd().resolve()
    src_folders = [folder / "src" for folder in [here, *here.parents]]
    matches = [folder for folder in src_folders if (folder / "mlfin_data.py").is_file()]
    sys.path.insert(0, str(matches[0]) if matches else "../src")
from mlfin_data import resolve_data_path

import warnings
import numpy as np                 # arrays of numbers
import pandas as pd                # tables with columns
import matplotlib.pyplot as plt    # charts
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.datasets import load_digits          # scikit-learn's handwritten digits: no file
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor   # scikit-learn's plain networks
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
plt.rcParams["font.size"] = 18     # big text for the projector

# Ames, Iowa home sales, 2006-2010 (Dean De Cock, Journal of Statistics Education, 2011).
homes = pd.read_csv(resolve_data_path("ames_housing.csv"))
log_price = np.log(homes.pop("SalePrice"))     # log dollars: an error is about a percent

# Kenneth French's 25 portfolios formed on size and book-to-market (value-weighted), with
# the market factor and the T-bill, per month, July 1963 to December 2025.
# Data: Copyright Eugene F. Fama and Kenneth R. French (French's Data Library).
# Returns are SIMPLE, in decimals per month (0.01 = 1%). mkt_rf: the market's EXCESS
# return (over the T-bill, rf). The portfolios are RAW returns: subtract rf for excess.
french = pd.read_csv(resolve_data_path("size_value_monthly.csv"))
french.index = pd.to_datetime(french["yyyymm"].astype(str), format="%Y%m")

# The Goyal-Welch monthly file of week 2. equity = stocks minus T-bills, simple, in decimals.
monthly = pd.read_csv(resolve_data_path("gw_monthly.csv"))
monthly.index = pd.to_datetime(monthly["yyyymm"].astype(str), format="%Y%m")
print(homes.shape, french.shape, monthly.shape)

Then run this cell too, every session: helpers that later parts use, so
that each part also runs on its own after these two cells.

In [ ]:
def make_curve():
    """The curve of "Hockey Sticks Add Up to Curves": a curved truth, sin(3x), and 300 noisy
    observations of it, x between -1 and 1. The same draw on every run (start 5)."""
    generator = np.random.default_rng(5)
    x = np.sort(generator.uniform(-1, 1, 300))
    truth = np.sin(3 * x)
    y = truth + generator.normal(0, 0.3, 300)
    return x, truth, y


def fit_digit_network():
    """The digits network of "A Picture Is a Grid of Numbers", fitted again: three quarters of
    the digits to fit. Returns the quarter it never sees (the digits and their labels) and the
    fitted network."""
    digits = load_digits()
    digits_fit, digits_new, labels_fit, labels_new = train_test_split(
        digits.data, digits.target, test_size=0.25, random_state=0)
    return digits_new, labels_new, MLPClassifier(random_state=0).fit(digits_fit, labels_fit)


def walk_downhill(step, epochs=60, neurons=20, start=1):
    """Plain gradient descent (new weight = old - step x blame) for one hidden layer on the
    curve's 300 points. Every step uses every row, so a step is an epoch. Returns the miss on
    the 300 points before the first step and after each epoch; stops once it passes a million."""
    x, _, y = make_curve()                         # the points; the true curve is not needed
    generator = np.random.default_rng(start)
    edge = np.sqrt(6 / (1 + neurons))              # small random starting weights
    w_in, b_in = generator.uniform(-edge, edge, neurons), generator.uniform(-edge, edge, neurons)
    w_out, b_out = generator.uniform(-edge, edge, neurons), generator.uniform(-edge, edge)
    misses = []
    for epoch in range(epochs + 1):
        sums = np.outer(x, w_in) + b_in                    # add
        outs = np.maximum(0, sums)                         # bend
        gap = outs @ w_out + b_out - y
        misses.append(np.mean(gap ** 2))
        if epoch == epochs or misses[-1] > 1e6:            # done, or run away
            break
        blame = 2 * gap / len(x)                           # the blame at each row's forecast
        blame_sums = np.outer(blame, w_out) * (sums > 0)   # passed back: x 1 if on, x 0 if off
        w_out, b_out = w_out - step * outs.T @ blame, b_out - step * blame.sum()
        w_in, b_in = w_in - step * x @ blame_sums, b_in - step * blame_sums.sum(axis=0)
    return np.array(misses)


# The dials' few rows: 20 of the curve's 300 points (every 15th) to fit, the other 280 held out.
curve_x, _, curve_y = make_curve()
every_15th = np.arange(300) % 15 == 0
few_X, few_y = curve_x[every_15th].reshape(-1, 1), curve_y[every_15th]
held_X, held_y = curve_x[~every_15th].reshape(-1, 1), curve_y[~every_15th]


def few_network(layers, penalty=0.0, start=0):
    """A network fitted to the 20 rows: a step of 0.01, 600 epochs with no stopping rule, and
    the penalty λ (scikit-learn's alpha), none unless told."""
    network = MLPRegressor(hidden_layer_sizes=layers, alpha=penalty, learning_rate_init=0.01,
                           max_iter=600, tol=0, n_iter_no_change=601, random_state=start)
    with warnings.catch_warnings():               # 600 epochs on purpose: no "not converged"
        warnings.simplefilter("ignore", ConvergenceWarning)
        return network.fit(few_X, few_y)


def fitting_miss(network):
    """The average squared gap between a network's forecasts and the 20 points it fitted."""
    return np.mean((network.predict(few_X) - few_y) ** 2)


def held_out_miss(network):
    """The average squared gap between a network's forecasts and the 280 held-out points."""
    return np.mean((network.predict(held_X) - held_y) ** 2)

## What a Network Does

| Application | Inputs → output |
|----------------------|--------------------------------------------------|
| ZIP codes, checks | a handwritten digit’s pixels → which digit |
| Medical images | a photo of the eye → signs of diabetic eye disease |
| Speech to text | sound → the words spoken |
| Translation | a Chinese sentence → the English one |
| ChatGPT | the words so far → the next word |
| AlphaGo (2016) | a Go board → a good next move |
| AlphaFold | a protein’s chain of amino acids → its 3-D shape |
| Card fraud alerts | a card payment → fraud or not |
| Gu, Kelly & Xiu (2020) | features (firm characteristics) → next month’s return |

- **Every row: one flexible function** of the inputs, fitted to make the
  **miss** small.
- **The last row is finance.** We come back to it in detail.

## A Picture Is a Grid of Numbers

- **scikit-learn’s built-in digits** (`load_digits`): small handwritten
  digits, each a grid of squares of ink, from white to black.
- **So a picture is a list of numbers,** the inputs. The output: which
  digit.
- **A plain network** (`MLPClassifier`: every square feeds every neuron)
  reads almost all of the digits it never saw.
- **Yann LeCun’s networks** (Bell Labs) read ZIP codes, then bank checks
  from 1993: over 10% of US checks by the late 1990s.

**Try it.** The whole network is one line, `MLPClassifier().fit(...)`:
scikit-learn’s plain network, one hidden layer of 100 neurons. It learns
from three quarters of the digits and is scored on the quarter it never
saw.

In [ ]:
# scikit-learn's built-in handwritten digits: no file to upload. Each digit is a grid of
# 8 x 8 squares, each square's ink from 0 (white) to 16 (black). MLPClassifier: scikit-learn's
# plain network for classes (imported in Setup).
digits = load_digits()
print(digits.images[0].astype(int))                # the first digit, a 0, as numbers

# digits.data: each grid as one row of 64 numbers; digits.target: which digit it is.
digits_fit, digits_new, labels_fit, labels_new = train_test_split(
    digits.data, digits.target, test_size=0.25, random_state=0)
digit_network = MLPClassifier(random_state=0).fit(digits_fit, labels_fit)
print(f"Share of new digits read right: {digit_network.score(digits_new, labels_new):.0%}")

## Convolution: Slide a Small Detector

- **A detector:** a small grid of **weights** (numbers a network learns)
  that score one pattern, such as an edge where white turns to ink.
- **Slide it over every window,** the same weights: the result, a map,
  is high wherever the pattern is.
- **A convolutional network** learns its detectors, stacked in layers:
  edges, then strokes, then digits. LeCun’s LeNet (1989, 1998); the 2018
  Turing Award, with Hinton and Bengio.

**Try it.** Slide the detector over the 0 by hand: two loops, one window
at a time. Then try another detector.

In [ ]:
# A convolution by hand. The detector: -1 on the left column, +1 on the right, so it is high
# where white turns to ink going right (the left edge of a stroke).
detector = np.array([[-1, 0, 1],
                     [-1, 0, 1],
                     [-1, 0, 1]])
zero = digits.images[0].astype(int)                # the first digit, a 0: 8 x 8 numbers
edge_map = np.zeros((6, 6), dtype=int)             # one number per 3 x 3 window
for row in range(6):                               # slide the window down ...
    for col in range(6):                           # ... and across
        window = zero[row:row + 3, col:col + 3]
        edge_map[row, col] = np.sum(window * detector)   # multiply square by square, add
print(edge_map)
# Try [[-1, -1, -1], [0, 0, 0], [1, 1, 1]]: high where white turns to ink going down.

## A Short History

- **1958:** the perceptron (Rosenblatt). **1969:** one layer’s limits
  (Minsky & Papert). **1986:** backpropagation (Rumelhart, Hinton &
  Williams). **1989:** universal approximation (Cybenko; Hornik,
  Stinchcombe & White). **1990s:** networks read handwritten checks
  (LeCun). **1997:** the LSTM, networks with memory (Hochreiter &
  Schmidhuber). **2012:** AlexNet wins at images, on graphics chips.
  **2017:** the transformer (Vaswani et al.). **2022:** ChatGPT.
- **The ideas are old.** What is new: the data (millions of labeled
  images, then the text of the web), graphics chips, and better tricks
  for training.
- **Two quiet spells for networks** in between, when other methods won.
- **Finance has less data:** about a hundred years of months for one
  market, and a tiny signal in each stock.

## Universal Approximation

- **1989, two proofs:** Cybenko; Hornik, Stinchcombe & White. Halbert
  White was an econometrician.
- **In plain words:** a network with one **hidden layer** (one layer of
  simple units, **neurons**, between inputs and output) can come as
  close as you like to any continuous curve over a bounded range, given
  enough neurons.
- **Leshno et al. (1993):** the same for almost any shape a neuron uses,
  including the simple one modern networks are built from.
- **It does not promise:**
  - how many neurons, or how much data;
  - that training finds the right weights;
  - anything outside the range of the data.
- **We come back to it** once neurons are built.

# One Hidden Layer

**Starting here?** Run the Setup cells at the top first, then this cell.
It rebuilds what this part uses from earlier parts: the digits network
of “A Picture Is a Grid of Numbers”.

In [ ]:
digits_new, labels_new, digit_network = fit_digit_network()

## One Neuron: Add, Then Bend

In [ ]:
# One neuron: a weighted sum of the features plus a bias (add), then max(0, z) (bend). The
# weights and the bias are made up, to show the arithmetic; a network learns them from data.
weights = {"size": -0.4, "value": 0.6, "momentum": 0.5}
bias = 0.1                                            # the neuron's intercept
stocks = {"stock A": {"size": -0.5, "value": 0.8, "momentum": 0.2},
          "stock B": {"size": 0.5, "value": -0.8, "momentum": -0.2}}
sums = {name: bias + sum(weights[key] * value for key, value in row.items())
        for name, row in stocks.items()}              # the "add" step
outputs = {name: max(0, z) for name, z in sums.items()}   # the "bend" step

z_grid = np.linspace(-1.2, 1.2, 100)
plt.figure(figsize=(10, 2.3))
plt.plot(z_grid, np.maximum(0, z_grid), color="tab:orange", linewidth=3)
for name, z in sums.items():
    plt.scatter(z, outputs[name], s=120, color="black", zorder=3)
    plt.text(z, outputs[name] + 0.2, name, ha="center")
plt.xlabel("z: the weighted sum")
plt.ylabel("neuron output")
for side in ["top", "right"]:
    plt.gca().spines[side].set_visible(False)
plt.show()

- **Add:**
  $z = 0.1 - 0.4\,\text{size} + 0.6\,\text{value} + 0.5\,\text{momentum}$.
- **Weights and biases:** −0.4, 0.6 and 0.5 are the **weights**, like a
  regression’s coefficients; 0.1 is the **bias**, like its intercept.
  Training chooses them.
- **Bend:** output $= \max(0, z)$, the **ReLU** (the bend is called the
  **activation**): a negative sum is cut to zero, and the neuron is off.

## A Network Is Stacked Regressions

- **Each hidden neuron:** a weighted sum of the inputs, like a
  regression’s forecast, then the bend: a hockey stick.
- **The forecast:** a weighted sum of the neurons.
- **Each link is one weight** (every input to every neuron, every neuron
  to the forecast), and each neuron also has a bias; all are fitted
  together, not one regression at a time.
- **Without the bend,** the stack collapses to one regression: the bends
  do the work.
- **Deep:** several hidden layers, stacked.

**Try it.** The digits network from above, opened up: how many weights
each layer of links holds, and how many biases.

In [ ]:
# A fitted network keeps its weights in coefs_ (one table per layer of links: every input to
# every neuron) and its biases in intercepts_ (scikit-learn's name; one per neuron).
for layer_weights in digit_network.coefs_:
    links_from, links_to = layer_weights.shape
    print(f"from {links_from} to {links_to}: {layer_weights.size:,} weights")
bias_count = sum(layer.size for layer in digit_network.intercepts_)
print(f"biases: {bias_count}")
weight_count = sum(layer.size for layer in digit_network.coefs_)
print(f"in all: {weight_count + bias_count:,} numbers, every one chosen in training")

## Hockey Sticks Add Up to Curves

In [ ]:
# The curve of the trees video ("One Round of Boosting"): one input x, a curved truth, and
# 300 noisy observations of it.
curve_generator = np.random.default_rng(5)
x_curve = np.sort(curve_generator.uniform(-1, 1, 300))
truth_curve = np.sin(3 * x_curve)
y_curve = truth_curve + curve_generator.normal(0, 0.3, 300)
X_curve = x_curve.reshape(-1, 1)                  # scikit-learn wants a table: one column


def miss(fitted):
    """How far a fitted curve is from the truth: the average squared gap."""
    return np.mean((fitted - truth_curve) ** 2)


line_fit = LinearRegression().fit(X_curve, y_curve).predict(X_curve)
tree_fit = DecisionTreeRegressor(max_depth=3).fit(X_curve, y_curve).predict(X_curve)
# Four hockey sticks, max(0, x - bend), with the bends FIXED by us at -0.75, -0.25, 0.25
# and 0.75; OLS chooses how much of each to add up.
bends = [-0.75, -0.25, 0.25, 0.75]
sticks = np.column_stack([np.maximum(0, x_curve - bend) for bend in bends])
sticks_fit = LinearRegression().fit(sticks, y_curve).predict(sticks)

figure, axes = plt.subplots(1, 3, figsize=(11, 2.6), sharey=True)
panels = [("line", line_fit, "tab:blue"), ("tree", tree_fit, "tab:green"),
          ("sticks", sticks_fit, "tab:orange")]
for axis, (name, fitted, color) in zip(axes, panels):
    axis.scatter(x_curve, y_curve, color="lightgray", s=10)
    axis.plot(x_curve, truth_curve, color="black", linestyle="--", linewidth=2)
    axis.plot(x_curve, fitted, color=color, linewidth=3)
    axis.set_title(f"{name}: miss {miss(fitted):.3f}")
    axis.set_xticks([])
    axis.set_yticks([])
for bend in bends:
    axes[2].axvline(bend, color="gray", linestyle=":", linewidth=1.5)
plt.subplots_adjust(wspace=0.05)
plt.show()

- **Miss:** the average squared error against the true curve (dashed),
  known only because we simulated it. Lines, then steps, then bends.
- Four hockey sticks, max(0, x − bend), bends fixed by us (dotted),
  weights from OLS: about as close as the tree. A network **chooses the
  bends itself**.

## More Sticks, Closer Fit

- **“Universal Approximation”, in one sentence:** more sticks, closer to
  the true curve, inside the range of the data.

**Try it.** More sticks, the same 300 points: the bends sit at the
middles of equal pieces of the range, and OLS adds the sticks up. Then
try other counts.

In [ ]:
# 2, 5 and 20 hockey sticks. The miss is against the true curve, as in the last cell.
for stick_count in [2, 5, 20]:                     # try 3, 50 or 200
    stick_bends = [-1 + (2 * piece + 1) / stick_count for piece in range(stick_count)]
    stick_table = np.column_stack([np.maximum(0, x_curve - bend) for bend in stick_bends])
    stick_fit = LinearRegression().fit(stick_table, y_curve).predict(stick_table)
    print(f"{stick_count:>3} sticks: miss {miss(stick_fit):.3f}")

## Check: No Bend

- A network with one hidden layer of 100 neurons, but no bend: each
  neuron passes its weighted sum straight on.
- **Fingers up:** it is the same as:
  1.  A linear regression
  2.  A curve with 100 bends
  3.  A tree with 100 leaves
  4.  The historical mean

**The check, by computer.** Vote first, then run it: the network of the
check, 100 neurons with no bend (`activation="identity"`), on the 300
points of the curve, next to OLS’s line.

In [ ]:
# 100 neurons, no bend: each neuron passes its weighted sum straight on.
no_bend_network = MLPRegressor(hidden_layer_sizes=(100,), activation="identity",
                               learning_rate_init=0.01, max_iter=5000, random_state=0)
no_bend_fit = no_bend_network.fit(X_curve, y_curve).predict(X_curve)
plt.figure(figsize=(10, 3))
plt.scatter(x_curve, y_curve, color="lightgray", s=8)
plt.plot(x_curve, no_bend_fit, color="tab:orange", linewidth=5, label="100 neurons, no bend")
plt.plot(x_curve, line_fit, color="black", linestyle="--", linewidth=2, label="OLS")
plt.legend(frameon=False)
plt.show()

# How a Network Learns

**Starting here?** Run the Setup cells at the top first, then this cell.
It rebuilds what this part uses from earlier parts: the curve of “Hockey
Sticks Add Up to Curves”, its 300 points and the true curve.

In [ ]:
x_curve, truth_curve, y_curve = make_curve()
X_curve = x_curve.reshape(-1, 1)                  # scikit-learn wants a table: one column

## Training: The Same Problem as OLS

|             | OLS                    | A network                     |
|:------------|:-----------------------|:------------------------------|
| Chooses     | the coefficients       | the weights and biases        |
| To minimize | the mean squared error | the same, or another **loss** |
| Solved by   | one formula            | an algorithm, step by step    |

- **Still an optimization problem:** choose the numbers that make the
  loss smallest. For yes or no, as for the logit, the loss punishes a
  low probability on what happened.
- **Too hard for a formula:** thousands of weights, bends between them,
  and a loss with many valleys.
- **The algorithm, repeated:**
  1.  Forecast with the current weights.
  2.  Compute the loss.
  3.  **Backpropagation:** each weight’s slope.
  4.  Move every weight a step downhill.

## Predict: A Step Too Big

- **One weight, to see the walk:** a portfolio’s **beta**, how much it
  moves when the market moves.
- **The walk:** start at beta = 0. At each step, move beta against the
  **slope** of the miss: how much the miss rises if beta rises a little.
  The **step size** sets how far.
- **Predict first:** what does a step that is much too big do?
  1.  Gets to the bottom faster
  2.  Overshoots, and runs away
  3.  Stops halfway
  4.  Never moves

## Walk Downhill

In [ ]:
# One weight to learn: a portfolio's beta. Small value stocks (ME1_BM5: the smallest fifth of
# firms, the highest fifth of book-to-market) and the market: monthly EXCESS returns in
# percent, July 1963 to December 2025.
portfolio = 100 * (french["ME1_BM5"] - french["rf"])
market = 100 * french["mkt_rf"]


def beta_miss(beta):
    """The miss for one beta: the average squared gap between the portfolio and beta x market."""
    return np.mean((portfolio - beta * market) ** 2)


def beta_slope(beta):
    """The slope of the miss at beta: positive means the miss rises as beta grows."""
    return -2 * np.mean(market * (portfolio - beta * market))


walks = {}
for step in [0.004, 0.02, 0.055]:
    beta = 0.0                                  # start at zero
    walks[step] = [beta]
    for move in range(8):                       # eight steps
        beta = beta - step * beta_slope(beta)   # the rule: a step against the slope
        walks[step].append(beta)
best_beta = np.mean(market * portfolio) / np.mean(market ** 2)   # the bottom, by formula

figure, axes = plt.subplots(1, 3, figsize=(11, 2.7))
for axis, (step, verdict) in zip(axes, [(0.004, "too small"), (0.02, "about right"),
                                        (0.055, "too big")]):
    path = walks[step]
    grid = np.linspace(min(path + [best_beta]) - 0.3, max(path + [best_beta]) + 0.3, 200)
    axis.plot(grid, [beta_miss(beta) for beta in grid], color="gray", linewidth=2)
    axis.plot(path, [beta_miss(beta) for beta in path], color="tab:orange", marker="o",
              markersize=7, linewidth=1.5)
    axis.axvline(best_beta, color="black", linestyle=":", linewidth=1.5)
    axis.set_title(f"{step}: {verdict}", fontsize=17)
    axis.set_xlabel("beta, zoomed out" if step == 0.055 else "beta")
    axis.set_yticks([])
axes[0].set_ylabel("miss")
plt.subplots_adjust(wspace=0.08)
plt.show()

- **The rule:** new beta = beta − step × slope of the miss. Start at 0,
  take 8 steps. The dotted line is the bottom.
- **Too small:** still short after 8 steps. **About right:** at the
  bottom within a few steps.
- **Too big:** each step overshoots by more than the last, and beta runs
  away.

## Backpropagation by Hand: Forward

- **One stock,** two features (value and momentum scores), and a network
  of two neurons, A and B, with made-up weights and no biases.
- **Add, then bend,** for A and for B; then add up the forecast and
  compare it with what happened: the miss.

## Predict: Which Weights Move?

- The network above (value 1, momentum 2; A on, B off): the forecast is
  below what happened.
- Neuron A is on; neuron B is off.
- One step downhill moves each weight against its own slope of the miss.
- **Predict first:** after one step, which weights have moved?
  1.  All six
  2.  The three on A’s path
  3.  Only the two into the forecast
  4.  None: the forecast is close enough

## Passing the Blame Back

- **Blame:** how much the miss moves when one number rises a little.
- **Backpropagation does it right to left, once:** a weight’s blame =
  the blame where its arrow lands × what traveled along it.
- **A bend:** × 1 if its neuron is on, × 0 if off. An off neuron stops
  the blame, so its weights get none.

## One Step Downhill, Every Weight

- **The rule of “Walk Downhill”,** for every weight at once: new = old −
  step × blame.
- **Big blame, big change; no blame, no change:** an off neuron’s
  weights stay put.
- **Forward again:** the forecast moves toward what happened, and the
  miss shrinks. A step too big overshoots.

**Try it.** The network of the last three slides in numpy: add, bend,
pass the blame back, and take one step for every weight, three times.
Then try a step of 0.2.

In [ ]:
# The two-neuron network of the last three slides. One stock: value score 1, momentum score 2;
# it returned 3%. into_neurons: one row per input, its arrows to A and to B. into_forecast: the
# arrows from A and from B to the forecast. No biases.
stock_scores = np.array([1.0, 2.0])
actual_return = 3.0
step_size = 0.05                                    # try 0.2
into_neurons = np.array([[0.2, 0.6],                # value -> A, value -> B
                         [0.4, -0.5]])              # momentum -> A, momentum -> B
into_forecast = np.array([1.0, 0.5])                # A -> forecast, B -> forecast
hand_misses = []
for step_number in range(4):
    neuron_sums = stock_scores @ into_neurons       # add ...
    neuron_outs = np.maximum(0, neuron_sums)        # ... then bend
    forecast = neuron_outs @ into_forecast
    hand_misses.append((actual_return - forecast) ** 2)
    print(f"steps so far: {step_number}; forecast {forecast:.2f}%, miss {hand_misses[-1]:.3f}")
    blame_forecast = 2 * (forecast - actual_return)                     # the blame at the forecast
    blame_neurons = blame_forecast * into_forecast * (neuron_sums > 0)  # passed back; x 0 if off
    # One step for every weight: new = old - step x blame (the blame where the arrow lands x
    # what traveled along it).
    into_forecast = into_forecast - step_size * blame_forecast * neuron_outs
    into_neurons = into_neurons - step_size * np.outer(stock_scores, blame_neurons)

## Batches and Epochs

- **A real network:** the same sweep, right to left, gives the blame of
  every weight at once, thousands or millions of them. That is
  **backpropagation**; software does the bookkeeping.
- **Rows:** each row has its own blames. A **batch** is a few rows
  (scikit-learn: 200): average their blames, take one step.
- **An epoch:** one pass through all the fitting rows.
- **Why batches:** one row at a time is jumpy; all rows at once is slow.
  The rows are reshuffled every epoch: **stochastic gradient descent**
  (all the slopes together: the **gradient**).

**Try it.** A network with one hidden layer of 100 neurons, trained one
epoch at a time on the curve’s 300 rows. `partial_fit` walks once
through the rows, in batches of 200: two steps an epoch. The miss on the
fitting rows falls over the epochs.

In [ ]:
# One epoch at a time: scikit-learn's walker (Adam), a step of 0.01, start 0.
epoch_network = MLPRegressor(hidden_layer_sizes=(100,), learning_rate_init=0.01, random_state=0)
for epoch in range(1, 51):
    epoch_network.partial_fit(X_curve, y_curve)                # one more epoch: two batches
    if epoch <= 5 or epoch % 10 == 0:
        epoch_miss = np.mean((epoch_network.predict(X_curve) - y_curve) ** 2)
        print(f"after epoch {epoch:>2}: miss {epoch_miss:.3f} on the fitting rows")

## Predict: Four Neurons or a Hundred?

- The curve of the hockey sticks: 300 points, one input.
- Two networks with one hidden layer: **4** neurons, and **100**.
- Each is trained three times, from three random starting weights.
- **Predict first:** which finds the curve from every start?
  1.  Four neurons
  2.  A hundred neurons
  3.  Both
  4.  Neither

## A Network Finds the Bends Itself

In [ ]:
# Two networks with one hidden layer, each trained from three random starting weights
# (random_state). learning_rate_init sets the size of the steps ("How a Network Learns").
network_fits = {}
for neurons in [4, 100]:
    for start in [0, 1, 2]:
        network = MLPRegressor(hidden_layer_sizes=(neurons,), learning_rate_init=0.01,
                               max_iter=5000, random_state=start)
        network.fit(X_curve, y_curve)
        network_fits[neurons, start] = network.predict(X_curve)

figure, axes = plt.subplots(1, 2, figsize=(11, 2.8), sharey=True)
for axis, neurons in zip(axes, [4, 100]):
    axis.scatter(x_curve, y_curve, color="lightgray", s=10)
    axis.plot(x_curve, truth_curve, color="black", linestyle="--", linewidth=2)
    for start, color in zip([0, 1, 2], ["tab:orange", "tab:blue", "tab:green"]):
        axis.plot(x_curve, network_fits[neurons, start], color=color, linewidth=2.5)
    axis.set_title(f"{neurons} neurons: three starts")
    axis.set_xticks([])
    axis.set_yticks([])
plt.subplots_adjust(wspace=0.05)
plt.show()

- **Four neurons:** the start decides: one start finds the curve, the
  others get stuck.
- **A hundred:** every start finds it, closer than the tree of depth 3.
- **No wiggles:** 300 points on one input are plenty. With few rows,
  networks do memorize: the dials, next.

## The Dials

| Dial | Too little | Too much |
|---------------------------|-----------------------|-----------------------|
| The step (learning rate) | creeps | overshoots, runs away |
| Neurons and layers | misses bends; starts get stuck | slow; can memorize |
| How long to train (epochs) | stops short | fits the noise |
| The penalty λ, as in ridge (`alpha` in scikit-learn) | weights chase noise | flattens the fit |
| Random starts, averaged | one start may be stuck | costs only time |
| Batch size | each step jumpy | slow steps |

- One section per dial next; batch size was “Batches and Epochs”.
- Choose every dial on **validation data**, never on the test rows.

## Dial: The Step

- **A training curve:** the miss on the fitting rows after each epoch.
- **Too small:** still creeping. **About right:** falls fast, then
  flattens.
- **Too big:** down once, then up, and it runs away: each step
  overshoots by more.
- In scikit-learn: `learning_rate_init`.

**Try it.** The training curves of this slide, as numbers.
`walk_downhill`, in the second Setup cell, is the plain walk downhill:
20 neurons on the curve’s 300 points, every row in every step. Then try
a step of 0.3: the miss jumps up and down, but does not run away.

In [ ]:
# Three step sizes, the same start: the miss before the first step and at the end.
for walk_step in [0.01, 0.1, 1.5]:
    step_misses = walk_downhill(walk_step)          # the miss at the start, then after each epoch
    print(f"step {walk_step}: miss {step_misses[0]:.2f} at the start, "
          f"{step_misses[-1]:,.2f} after {len(step_misses) - 1} epochs")

## Dial: Neurons and Layers

- **Few rows:** 20 of the curve’s 300 points (every 15th) to fit; the
  other 280 held out. The miss is on the held-out points (against the
  points, not the true curve: never below the noise).
- **Too few** neurons: too stiff, misses the bends. **Too many:** they
  hold far more numbers than there are points, and draw through every
  point.
- **A layer** bends the bends of the layer before; the count of numbers
  grows with width × width.
- More neurons: fewer stuck starts, but slower to train.

**Try it.** The three networks of this slide on the dials’ 20 points:
`few_network`, `fitting_miss` and `held_out_miss` are in the second
Setup cell. The 2 layers of 100 memorize the 20 points, so their misses
can differ a little, in the third decimal, from the slide’s and from one
computer to another. Then try `(5,)` or `(100,)`.

In [ ]:
# Three sizes, each trained 600 epochs on the 20 points from start 0, no penalty.
for hidden_layers in [(2,), (20,), (100, 100)]:
    size_network = few_network(hidden_layers)
    print(f"{str(hidden_layers):>10}: held-out miss {held_out_miss(size_network):.3f}, "
          f"fitting miss {fitting_miss(size_network):.3f}")

## Dial: How Long to Train

- **The gap opens:** the fitting miss falls toward zero; the held-out
  miss falls, then climbs.
- **Early stopping:** hold out 10% of the fitting rows; stop when their
  miss stops improving. For returns, hold out the latest months.
- In scikit-learn: `early_stopping=True`; it waits a few epochs without
  improvement, then keeps the best epoch’s weights.

**Try it.** The 2 layers of 100 of the last cell, trained one epoch at a
time (`partial_fit`) and scored after some epochs on the 20 points it
fits and on the 280 held out. Past the best epoch, its numbers can
differ a little from the slide’s.

In [ ]:
# The 2 layers of 100, no penalty, start 0: one epoch at a time, 600 epochs.
long_network = MLPRegressor(hidden_layer_sizes=(100, 100), alpha=0.0, learning_rate_init=0.01,
                            random_state=0)
for epoch in range(1, 601):
    long_network.partial_fit(few_X, few_y)                      # one more epoch
    if epoch in [10, 39, 100, 300, 600]:
        print(f"epoch {epoch:>3}: fitting miss {fitting_miss(long_network):.3f}, "
              f"held-out miss {held_out_miss(long_network):.3f}")

## Dial: The Penalty

- **Ridge’s penalty, on every weight:** the squared gaps plus λ × the
  sum of squared weights. It pulls the weights toward zero; the curve
  calms down.
- **No penalty:** through every point. **A middle one:** smooth. **A big
  one:** almost a straight line.
- In scikit-learn: `alpha`. Search it over powers of ten, on validation
  data.

**Try it.** The same 2 layers of 100 at the three penalties of this
slide. λ = 0 is the memorizing network, so its miss can differ a little
from the slide’s. Then try 0.01 or 1.

In [ ]:
# The 2 layers of 100 on the 20 points, 600 epochs, at three penalties λ (alpha in scikit-learn).
for penalty_value in [0.0, 0.1, 3.0]:
    penalty_network = few_network((100, 100), penalty=penalty_value)
    print(f"λ = {penalty_value:g}: held-out miss {held_out_miss(penalty_network):.3f}")

## Dial: Random Starts

- **With few neurons the start decides:** the same network, from another
  random start, misses by twice as much.
- **Averaging the starts** helps where they disagree, barely where they
  agree: it removes the start’s noise, not the data’s.

**Try it.** 10 neurons on the 20 points from five random starts, each
scored on the 280 held-out points, then the five forecasts averaged.
Then try 20 neurons.

In [ ]:
# 10 neurons, five random starts (random_state), 600 epochs each, no penalty.
forecasts_by_start = []
for start in range(5):
    start_network = few_network((10,), start=start)
    forecasts_by_start.append(start_network.predict(held_X))
    print(f"start {start}: held-out miss {held_out_miss(start_network):.3f}")
average_forecast = np.mean(forecasts_by_start, axis=0)        # the five forecasts, averaged
print(f"the five averaged: held-out miss {np.mean((average_forecast - held_y) ** 2):.3f}")

## Why Tuning a Network Is Harder

| Model         | Dials that matter |
|---------------|-------------------|
| Random forest | 2                 |
| Boosted trees | 3                 |
| Network       | 6                 |

- **More dials, multiplied:** neurons, layers, step, epochs, λ, batch
  size, and the starts. Each fit is slower than a tree’s. A forest does
  well out of the box.
- **They interact:** a bigger network wants a smaller step, fewer epochs
  or a larger λ. Change one, and the best value of another moves.
- **The start moves the score** as much as a dial does.
- **In practice:** fix most dials; tune two or three on validation data.

## Check: A Jumpy Miss

- A network’s miss on its fitting rows, printed after every epoch, jumps
  up and down, and grows.
- **Fingers up:** which dial do you change first?
  1.  Lower the learning rate
  2.  Add neurons
  3.  Train longer
  4.  Add a layer

# Networks on Real Data

## The Network on the Homes

**The race of the trees deck, and a network joins it.** The homes: 1,460
sales, 79 features, log price. The first cell below prepares the race,
the second runs it and prints each model’s $R^2$ on homes it did not see
(about a minute on Colab). Then a network joins: scikit-learn’s default
size and penalty (one hidden layer of 100 neurons), early stopping on,
nothing tuned; three random starts, their forecasts averaged.

In [ ]:
# The race of the trees deck ("The Race on Real Data"): five models, each scored on homes it
# did not see (5-fold cross-validation). This cell prepares it and takes no time; the next cell
# runs it. The on-your-own cell after "One Start Cannot Choose" needs this cell and the one
# that defines homes_network, not the race.
# A blank in a feature column (no alley, no pool) means "none", so categories get "none".
word_columns = homes.select_dtypes(exclude="number").columns.tolist()
number_columns = [name for name in homes.columns if name not in word_columns]

dummies = make_pipeline(SimpleImputer(strategy="constant", fill_value="none"),
                        OneHotEncoder(handle_unknown="ignore"))
for_lines = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), number_columns),
    ("words", dummies, word_columns)])
for_trees = ColumnTransformer([
    ("numbers", SimpleImputer(strategy="median"), number_columns), ("words", dummies, word_columns)])
folds = KFold(5, shuffle=True, random_state=0)


def homes_r2(forecasts):
    """R², %, of out-of-fold forecasts of log price: the forecast for each home comes from a
    model that never saw that home."""
    return 100 * (1 - np.sum((log_price - forecasts) ** 2)
                  / np.sum((log_price - log_price.mean()) ** 2))

In [ ]:
# The race itself: about a minute on Colab.
racers = {
    "linear regression": make_pipeline(for_lines, LinearRegression()),
    "ridge": make_pipeline(for_lines, RidgeCV(alphas=np.logspace(-1, 3, 30))),
    "one tree, depth 4": make_pipeline(for_trees, DecisionTreeRegressor(
        max_depth=4, min_samples_leaf=10, random_state=0)),
    "random forest": make_pipeline(for_trees, RandomForestRegressor(
        n_estimators=200, min_samples_leaf=2, max_features=0.33, n_jobs=-1, random_state=0)),
    "boosted trees": make_pipeline(for_trees, GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.03, max_depth=3, subsample=0.8, random_state=0)),
}

homes_forecasts = {}                         # each model's out-of-fold forecast of log price
for name, racer in racers.items():
    homes_forecasts[name] = cross_val_predict(racer, homes, log_price, cv=folds)
homes_score = {name: homes_r2(forecast) for name, forecast in homes_forecasts.items()}
for name, score in homes_score.items():
    print(f"{name:18s} R² {score:5.1f}%")

In [ ]:
# The network that joins the race: defined here, run in the next cell and on your own later.
def homes_network(start, layers=(100,), penalty=0.0001):
    """A network for log price: scikit-learn's default unless told otherwise (one hidden layer
    of 100 neurons; penalty λ = 0.0001, which scikit-learn calls alpha). Inputs prepared as for
    ridge (for_lines: numbers standardized, categories as 0/1 columns). early_stopping: hold
    out 10% of the fitting rows and stop when their miss stops improving.
    TransformedTargetRegressor standardizes log price inside each fit and turns the forecasts
    back into log price: a network starts near zero, and log price is about 12."""
    network = MLPRegressor(hidden_layer_sizes=layers, alpha=penalty, early_stopping=True,
                           random_state=start)
    return TransformedTargetRegressor(make_pipeline(for_lines, network),
                                      transformer=StandardScaler())

In [ ]:
start_forecasts = {}                       # the same 5 folds as the race; a few seconds
for start in [0, 1, 2]:
    start_forecasts[start] = cross_val_predict(homes_network(start), homes, log_price,
                                               cv=folds)
start_scores = {start: homes_r2(forecast) for start, forecast in start_forecasts.items()}
network_score = homes_r2(np.mean(list(start_forecasts.values()), axis=0))   # average of 3

rows = {**homes_score, "network, average of 3": network_score}
rows["network, each start"] = np.mean(list(start_scores.values()))      # for the order only
order = sorted(rows, key=rows.get)                                       # lowest at the bottom
colors = {"linear regression": "gray", "ridge": "gray", "one tree, depth 4": "tab:blue",
          "random forest": "tab:orange", "boosted trees": "tab:orange"}
plt.figure(figsize=(10, 3.4))
for position, name in enumerate(order):
    if name == "network, each start":         # three dots, one per start
        low, high = min(start_scores.values()), max(start_scores.values())
        plt.plot([low, high], [position, position], color="tab:green", linewidth=2)
        plt.scatter(list(start_scores.values()), [position] * 3, color="tab:green", s=60, zorder=3)
        plt.text(high + 0.5, position, f"{low:.1f} to {high:.1f}%", va="center")
    else:
        plt.barh(position, rows[name], color=colors.get(name, "tab:green"))
        plt.text(rows[name] + 0.5, position, f"{rows[name]:.1f}%", va="center")
plt.yticks(range(len(order)), order)
plt.xlim(60, 100)
plt.xlabel("R² on homes the model did not see (%)")
plt.show()

- **A network joins the trees deck’s race:** default size and penalty,
  early stopping on, nothing tuned. The start alone moves the score.
- **Their average** beats every start and the forest; boosting stays
  ahead.
- **79 features, 1,460 homes:** a table that suits trees. Networks shine
  with far more data and raw inputs (pictures, text).

## One Start Cannot Choose

- **Four changes, one start each** (16 or 256 neurons, two layers, λ =
  10): 256 neurons scores best, a little above the default.
- **The start alone** moves the default’s score by a similar amount: one
  start each cannot tell better from luckier.
- **The default’s three starts, averaged,** score more than any change.

**On your own, after class.** Change one setting and score the network
from one start, as on the slide. It needs the cell that prepares the
race (under “The Network on the Homes”) and the cell that defines
`homes_network`, not the race. About two minutes on Colab.

In [ ]:
# One start (start 0) of the network with your settings, scored on homes it did not see.
# From start 0: the default 85.0, (16,) 84.4, (64, 32) 86.6, (256,) 87.5, λ = 10 85.6.
my_layers = (256,)        # neurons per layer: try (16,), (64, 32) or (256,)
my_penalty = 0.0001       # the penalty λ (scikit-learn's alpha); 0.0001 is the default; try 10
my_network = homes_network(0, my_layers, my_penalty)
my_forecast = cross_val_predict(my_network, homes, log_price, cv=folds)
print(f"R² {homes_r2(my_forecast):.1f}%")

# A Finance Application

## Which Stocks Will Do Better?

- **Gu, Kelly & Xiu (2020):** can a network tell which stocks will do
  better next month?
- **One row:** one stock in one month. **The target:** its return the
  next month, over the T-bill.
- **Almost 30,000 US stocks,** 1957–2016: over 6,200 in a typical month.
- **94 features (firm characteristics):** size, value, momentum,
  volatility, trading volume, … Each month, each is ranked from −1 (the
  lowest stock) to +1 (the highest); a blank gets the middle, 0.
- **920 features in all:** each characteristic alone and times each of 8
  market-wide numbers (the T-bill rate, …), plus 74 industry dummies.
- **The score:** $R^2$ on years no model saw, against a forecast of
  **zero**: stricter than a stock’s own past average, which is so noisy
  that it is easy to beat.

## The Split in Time

- **Fit** on the past, **choose the dials** on the 12 years just before,
  **forecast** one year no model has seen.
- **Then move a year** and refit: one more fitting year; the 12
  validation years roll forward. 30 refits forecast 1987–2016.
- **No shuffling:** the rows keep their order in time. Early stopping
  watches the validation years.

## Five Networks, One Recipe

- **One to five hidden layers,** each half as wide as the last: 32, 16,
  8, 4, 2 neurons, all with the hockey-stick bend.
- **About 30,000 numbers** each, almost all of them between the 920
  features and the first 32 neurons.
- **The recipe:** the squared miss plus lasso’s penalty λ; Adam, a
  walker that adjusts each weight’s step as it goes; stop when the
  validation years stop improving; average 10 random starts.
- **Shapes fixed in advance;** only λ and the step were tuned. A search
  over shapes: “unrealistic and unnecessary”.

## Who Forecasts Best

- **OLS on all 920 features: −3.46%,** worse than forecasting zero.
  Penalties help, and so do PCR and PLS (regressions on a few
  combinations of the features); curves in each feature alone add
  little.
- **Trees and networks lead,** with interactions. Best: three layers,
  0.40%; four and five layers do no better.
- **Is the gap real?** Over OLS on three features, yes (two to five
  layers); over the trees, no: within the noise.

## What the Networks Use

- **Importance:** set one feature to the middle (0), keep the fitted
  model, and see how much $R^2$ falls.
- **All their models agree** on a short list: price trends, then
  liquidity, volatility, valuation. Linear models lean on one or two;
  trees and networks spread out.
- **It depends on size:** last month’s winners fall back and losers
  bounce, strongly for small stocks; for large stocks, mostly the
  winners fall back.

## Tiny R², Real Money

- **An $R^2$ of only 0.40%** (three layers), yet each month: sort the
  stocks into tenths by its forecast; buy the top tenth, sell the
  bottom.
- **Value-weighted** (bigger firms count more): Sharpe ratio **1.20** a
  year. Four layers did best, 1.35; OLS on size, value and momentum,
  0.61.
- **Why it pays:** one stock’s month is mostly noise; across a tenth of
  the market the noise averages out. **Before trading costs,** with
  turnover over 100% a month.

# Networks With Memory

## Tomorrow From the Past

**This part reads one more file:** download `market_daily.csv` from
Brightspace and drag it into the Files pane (folder icon on the left),
as with the other files.

In [ ]:
# The stock market's DAILY return over T-bills: French's daily file, July 1926 to December
# 2025. mkt_rf: the market's EXCESS return (over the T-bill, rf), simple, in decimals per day.
# Data: Copyright Eugene F. Fama and Kenneth R. French (French's Data Library).
daily = pd.read_csv(resolve_data_path("market_daily.csv"), index_col="date", parse_dates=True)
market = 100 * daily["mkt_rf"]["1966":"1985"]     # in percent; the years we fit, 1966 to 1985
today_tomorrow = np.corrcoef(market.values[:-1], market.values[1:])[0, 1]
print(f"{len(market):,} days; correlation of today's and tomorrow's return: {today_tomorrow:.2f}")

- **One series:** the stock market’s daily return over T-bills, from
  1966 to 1985, about 5,000 days.
- **The question:** forecast tomorrow’s return from the last 20 days.
  Which of the 20 matter: today? last week? all of them?
- **A network should find that out itself,** from the data.

## Lag Columns: You Pick the Past

| Forecast made on | Today | 1 back | 2 back |  …  | 19 back | Tomorrow |
|:-----------------|:-----:|:------:|:------:|:---:|:-------:|:--------:|
| Wed Oct 3, 1979  | Oct 3 | Oct 2  | Oct 1  |  …  |  Sep 6  |  Oct 4   |
| Thu Oct 4, 1979  | Oct 4 | Oct 3  | Oct 2  |  …  |  Sep 7  |  Oct 5   |
| …                |       |        |        |     |         |          |

- **A plain network reads one row at a time:** 20 numbers in, a forecast
  of tomorrow out. Nothing carries over: each row repeats the row above,
  one column over.
- **You choose the columns:** 20 days or 60? Raw days, or a 5-day
  average? The network builds no summary of the past on its own; it
  relies on you getting this right.

## A Running Summary: It Builds Its Own

| Day       | It reads       | It carries forward                              |
|:---------------|:------------------|:-------------------------------------|
| Wed Oct 3 | Oct 3’s return | state 1, made from that return                  |
| Thu Oct 4 | Oct 4’s return | state 2, made from that return and state 1      |
| Fri Oct 5 | Oct 5’s return | state 3, made from that return and state 2      |
| …         | …              | …                                               |
| Tue Oct 9 | Oct 9’s return | state 5, which makes the forecast for Wednesday |

- **A recurrent network reads one day at a time,** oldest first, and
  computes its own summary of the past: its **state**. “Recurrent”: the
  same step recurs every day.
- **Any number of days, one rule:** a big drop teaches the same weights
  whatever day it happened. Text the same way: a sentence of any length,
  word by word.

## One Rule, Applied Every Day

1.  **Day 1:** its return goes into the box; out comes the **state**, a
    note on the past.
2.  **Day 2:** its return and the state go in; out comes a new state.
3.  **Every day, the same rule:** state = **squash**(weight × last
    state + weight × today’s return + bias). The squash caps big sums
    between −1 and 1.
4.  **The last state makes the forecast:** a weight × the state + a
    bias.

## Unrolled: Five Days by Hand

- **Made-up weights, real days** (October 3 to 9, 1979): 0.8 on the last
  state, 0.5 on today’s return, a bias of 0; the forecast is 0.5 × the
  state.
- **Each day:** 0.8 × last state + 0.5 × today’s return, then squashed.
  A small sum passes the squash almost unchanged; a big one is capped.
- **The forecast for the day after the last:** 0.5 × the last state.

**Try it.** The network of this slide, day by day: one line in a loop,
the same weights every day. Then try 0.2 instead of 0.8 on the last
state: the state forgets the earlier days faster.

In [ ]:
# Made-up weights: 0.8 on the last state, 0.5 on today's return, a bias of 0; then the
# squash, tanh. The forecast for the next day: 0.5 x the state. Returns in percent.
state = 0.0                                          # before the first day
for day, one_return in zip(["Wed Oct 3", "Thu Oct 4", "Fri Oct 5", "Mon Oct 8", "Tue Oct 9"],
                           [0.13, 0.56, 0.85, -1.24, -3.44]):
    state = np.tanh(0.8 * state + 0.5 * one_return)       # the same weights every day
    print(f"{day}: return {one_return:+.2f}%, state {state:.2f}")
print(f"forecast for Wed Oct 10: {0.5 * state:+.2f}%")

## The State Is Learned

- **The weight on the last state decides what the state keeps:** a small
  one forgets fast, one near 1 keeps weeks. Ours was made up.
- **Training chooses it,** with the other weights, to make tomorrow’s
  forecast best: **the network decides what to remember.**
- **The state can be many numbers:** ours is one; real networks carry
  dozens (Fischer & Krauss: 25), each its own summary of the past.

## Training: Back Through the Days

In [ ]:
# Every run of 20 days in 1966-1985 is one window; the day after it is the return to forecast.
LENGTH = 20
windows = np.lib.stride_tricks.sliding_window_view(market.values[:-1], LENGTH)  # one row a window
tomorrow = market.values[LENGTH:]                                               # the day after each


def run_network(windows, weights):
    """Read every window, oldest day first; return the forecast for the day after each."""
    w_state, w_return, bias, w_out, bias_out = weights
    state = np.zeros(len(windows))                     # one state per window, 0 at the start
    for day in range(windows.shape[1]):
        state = np.tanh(w_state * state + w_return * windows[:, day] + bias)
    return w_out * state + bias_out


def train_network(windows, tomorrow, weights, step=0.1, steps=2000):
    """Walk downhill: each step runs every window forward, passes the blame back through the
    days (backpropagation through time) and moves all five numbers (three weights, two biases).
    Returns the new numbers and the weight on the last state after each step."""
    w_state, w_return, bias, w_out, bias_out = weights
    path = [w_state]
    for _ in range(steps):
        states = [np.zeros(len(windows))]              # forward, keeping every day's state
        for day in range(windows.shape[1]):
            states.append(np.tanh(w_state * states[-1] + w_return * windows[:, day] + bias))
        miss = w_out * states[-1] + bias_out - tomorrow
        blame = 2 * miss / len(tomorrow)               # the blame at each window's forecast
        blame_w_out, blame_bias_out = blame @ states[-1], blame.sum()
        blame_state = blame * w_out                    # passed back to the last day's state
        blame_w_state = blame_w_return = blame_bias = 0.0
        for day in range(windows.shape[1] - 1, -1, -1):        # back through the days
            blame_sum = blame_state * (1 - states[day + 1] ** 2)   # x the squash's slope
            blame_w_state += blame_sum @ states[day]   # the same weights every day: blame adds up
            blame_w_return += blame_sum @ windows[:, day]
            blame_bias += blame_sum.sum()
            blame_state = blame_sum * w_state          # one day further back
        w_state -= step * blame_w_state
        w_return -= step * blame_w_return
        bias -= step * blame_bias
        w_out -= step * blame_w_out
        bias_out -= step * blame_bias_out
        path.append(w_state)
    return (w_state, w_return, bias, w_out, bias_out), np.array(path)


print(f"{len(windows):,} windows of {LENGTH} days")

- **Forward:** read 20 days, oldest first; forecast the 21st; the miss
  is the squared gap to the actual return.
- **Back:** the miss’s blame passes back day by day, through the same
  box each time (“backpropagation through time”).
- **Same weights every day:** each weight’s blame adds up over the 20
  days. Then one step downhill, for all the windows at once.

## Which Past Does It Use?

In [ ]:
# Train from the made-up weights of "Unrolled: Five Days by Hand": 2,000 steps of 0.1. About
# 2 seconds. Then ask the network which days it leans on: raise one day's return by 1 point
# in every window and see how much the forecast moves, on average.
start_weights = (0.8, 0.5, 0.0, 0.5, 0.0)      # last state, today's return, bias; forecast, bias
trained_weights, state_weight_path = train_network(windows, tomorrow, start_weights)


def leans_on(weights, windows):
    """How much the forecast moves when one day's return is 1 point higher: today first."""
    base = run_network(windows, weights)
    moves = []
    for days_back in range(LENGTH):
        bumped = windows.copy()
        bumped[:, LENGTH - 1 - days_back] += 1         # that day's return, 1 point higher
        moves.append(np.mean(run_network(bumped, weights) - base))
    return np.array(moves)


lean_before, lean_after = leans_on(start_weights, windows), leans_on(trained_weights, windows)
print(f"weight on the last state: {start_weights[0]:.2f} before, {trained_weights[0]:.2f} after")
print("leans on today, yesterday, 2 days before:")
print("  before:", np.round(lean_before[:3], 2), " after:", np.round(lean_after[:3], 2))
# Regression on 20 lag columns (today first), for comparison.
lag_columns = np.column_stack([np.ones(len(windows)), windows[:, ::-1]])
lag_slopes = np.linalg.lstsq(lag_columns, tomorrow, rcond=None)[0][1:]
print("regression on 20 lag columns:", np.round(lag_slopes[:3], 2))

- **Before training** (the made-up weights): the forecast leans on the
  last week or two, fading.
- **After training:** it leans on today’s return, a little the other way
  on yesterday’s, and about not at all on the rest. The weight on the
  last state walked from 0.8 to below 0: it learned to forget fast.
- **The data’s own lesson:** a regression on 20 lag columns finds the
  same.

## Does the Lesson Last?

In [ ]:
# The network trained on 1966-1985, on the decades after, which it never saw. Each decade: the
# correlation of today's and tomorrow's return, and the network's R² out of sample: the share of
# the days' squared ups and downs its forecast explains, against forecasting the average day of
# the fitting years every day. About a second.
def windows_of(returns):
    """Every run of 20 days in a series, and the day after each."""
    return np.lib.stride_tricks.sliding_window_view(returns[:-1], LENGTH), returns[LENGTH:]


fit_average = tomorrow.mean()                       # the average day of 1966-1985, in percent
decades = [("1966", "1975"), ("1976", "1985"), ("1986", "1995"), ("1996", "2005"),
           ("2006", "2015"), ("2016", "2025")]
decade_correlation, decade_r2 = [], []
for first, last in decades:
    decade_returns = 100 * daily["mkt_rf"][first:last].values
    decade_correlation.append(np.corrcoef(decade_returns[:-1], decade_returns[1:])[0, 1])
    line = f"{first} to {last}: correlation {decade_correlation[-1]:+.2f}"
    if first > "1985":                              # a decade the network never saw
        later_windows, later_tomorrow = windows_of(decade_returns)
        later_forecast = run_network(later_windows, trained_weights)
        decade_r2.append(100 * (1 - np.sum((later_tomorrow - later_forecast) ** 2)
                                / np.sum((later_tomorrow - fit_average) ** 2)))
        line += f", the network's R² {decade_r2[-1]:+.1f}%"
    print(line)

- **Fitted on 1966–1985, scored on days it never saw:** in 1986–1995 its
  forecast beat the fitting years’ average day. For daily returns, a
  real forecast.
- **Then below zero:** the day-to-day correlation shrank, and since 2006
  it is negative: a rise today now tends to reverse a little.
- **One common reading:** many stocks once traded rarely, so news
  reached their prices a day late; as trading grew, that gap closed. So
  test a network on later years, and refit.

## The Data Decide the Memory

In [ ]:
# Simulated data: a world whose expected return drifts slowly. Each day it keeps 95% of the day
# before, plus a small shock; the day's return is that expected return plus a noise of 1 point
# (both in percent). Trained as on the market, then scored on new simulated days. About 2 seconds.
def slow_world(seed, days=len(market)):
    """Simulated daily returns and the expected returns behind them, in percent."""
    draw = np.random.default_rng(seed)
    expected = np.zeros(days)
    for day in range(1, days):
        expected[day] = 0.95 * expected[day - 1] + 0.125 * draw.standard_normal()
    return expected + draw.standard_normal(days), expected


slow_returns, slow_expected = slow_world(1)
slow_windows, slow_tomorrow = windows_of(slow_returns)
slow_weights, slow_path = train_network(slow_windows, slow_tomorrow, start_weights)  # as before
slow_lean = leans_on(slow_weights, slow_windows)
fresh_returns, _ = slow_world(2)                    # new simulated days, never seen
fresh_windows, fresh_tomorrow = windows_of(fresh_returns)
fresh_forecast = run_network(fresh_windows, slow_weights)
slow_r2 = 100 * (1 - np.sum((fresh_tomorrow - fresh_forecast) ** 2)
                 / np.sum((fresh_tomorrow - slow_tomorrow.mean()) ** 2))
print(f"weight on the last state: {slow_weights[0]:.2f} (on the market: {trained_weights[0]:.2f})")
print("leans on today, 5 and 10 days before:", np.round(slow_lean[[0, 5, 10]], 2))
print(f"R² on new simulated days: {slow_r2:.1f}%")

- **A simulated world with a slow memory:** each day’s expected return
  keeps 95% of the day before, so it drifts for weeks; each day’s return
  adds a much bigger noise.
- **The same network, start and training:** the weight on the last state
  walks from 0.8 up toward 1 (on the market, to below 0). It leans on
  many days, fading slowly.
- **On new simulated days** it explains part of the squared ups and
  downs. **The data, not we, decide what the state keeps.**

## Why Blame Fades

- **Each day back multiplies the blame** by the weight on the last state
  and by the squash’s slope, at most 1.
- **0.5 multiplied 20 times is about one millionth:** a big drop 20 days
  back teaches the weights almost nothing.
- **On the market no harm was done:** the lesson was today. The slow
  world’s weight near 1 let blame travel back, but such a state also
  forgets slowly after a shock.

## The LSTM in Words: Keep, Write, Show

| Made-up dials | Memory before | Today’s news | **Keep** | **Write** | Memory after |
|:---------------------|:--------:|:--------:|:--------:|:--------:|:--------:|
| A calm day | 0.60 | 0.10 | 0.95 | 0.10 | keep × before + write × news |
| A crash day | 0.60 | −0.90 | 0.30 | 0.90 | keep × before + write × news |

- **The memory:** a second running note, beside the state.
- **Three dials between 0 and 1, set fresh every day** from today’s
  return and yesterday’s state: how much of the memory to **keep**, how
  much of today’s news to **write**, how much to **show** the forecast.
- **Memory after = keep × memory before + write × today’s news.** The
  plain network kept a fixed share every day (0.95 in the slow world);
  the LSTM’s keep dial is near 1 when calm and drops after a crash.

## The LSTM: Memory With Gates

- **The memory line** (blue) changes only by × keep and + write × news:
  along it, blame can travel far back.
- **Each dial is a neuron** with an S-bend between 0 and 1, reading
  today’s return and the shown state. Textbook names: keep is the
  **forget gate**, write the **input gate**, show the **output gate**.
- **It fixes some of the plain network’s problems:** it can learn from
  weeks back, and keep a long memory yet react fast.

## Fischer & Krauss: The Setup

|  | **Our market network** | **Fischer & Krauss (2018)** |
|:----------------|:------------------------|:-----------------------------|
| Series | The market’s daily return | Each S&P 500 stock’s daily return |
| Days read | 20 | 240, about a year |
| Network | One neuron: 5 numbers | An LSTM of 25 cells: 2,752 numbers |
| Forecast | Tomorrow’s return | Will the stock beat the median stock tomorrow? |
| Fit, then test | 1966–1985, then later years | 3 years, then the next one; 23 times, 1992–2015 |

- **One network for all the stocks:** every stock’s 240 days pass
  through the same weights, as our days passed through the same box.
- **Each day:** buy the 10 stocks with the best chance, sell the 10 with
  the worst. Rivals: a random forest, a plain deep network and a logit,
  fed past returns over 1 to 240 days.

## Fischer & Krauss: What It Earned

- **Their headline: the LSTM beat every model without memory,** 0.46% a
  day before costs; a random forest 0.43%; a plain deep network and a
  logit well behind.
- **After costs** of 0.05% on each buy or sell: 0.26% a day; the forest
  0.23%, the plain deep network 0.12%.
- **Over time:** strong in the 1990s, smaller in the 2000s, about
  break-even after costs from 2010: their “time of deterioration”.

## Fischer & Krauss: What the LSTM Used

| The 10 stocks the LSTM… | **Last 5 days** | **Days 20 to 240 before** |
|:------------------------|:---------------:|:-------------------------:|
| bought each day         |      −4.1%      |           +9.1%           |
| sold each day           |      +4.5%      |           +7.8%           |
| the average stock       |      +0.3%      |          +11.7%           |

- **It bought stocks that had just fallen and sold those that had just
  risen** (“short-term reversal”): like our market network, it read a
  long history but leaned on the last few days.
- **A plain rule,** buy the 5-day losers and sell the 5-day winners,
  earns half as much, and almost nothing after costs: the network found
  this known pattern by itself, and as much again beyond it.

# What Networks Can Do That Trees Cannot

## Where Networks Changed the Game

- **Pictures:** in ImageNet’s contest (1.2 million labeled photos), the
  winner’s misses fell from 28% to 4% once networks took over.
- **Elsewhere:** speech and translation; Go (AlphaGo beat Lee Sedol,
  2016); proteins (AlphaFold, Nobel Prize in Chemistry 2024); ChatGPT.
- **In finance:** price charts read as pictures; news read as text; each
  stock’s daily returns read in order (Fischer & Krauss).

## Raw Inputs, Learned Features

- **Trees, and plain networks, see 64 unrelated columns.** Moved one
  pixel, a handwritten 0 is a new row: most of its 64 numbers change. On
  these tiny grids that hardly matters; on real photos it does.
- **A convolutional network** slides small detectors over the picture:
  it finds a stroke wherever it sits, then builds strokes into shapes.
- **Words, sound, price paths:** recurrent networks read sequences and
  learn what to carry forward. Nobody hand-builds these features; the
  network learns them.

**Try it.** The plain network of “A Picture Is a Grid of Numbers”,
fitted again, on the quarter of the digits it never saw. In the file,
every digit is written in about the same place, so the network reads
almost all of them. Move each one pixel to the right, as an object moves
in a photo, and it reads about half.

In [ ]:
# The digits network, fitted again (fit_digit_network is in the second Setup cell), and the
# digits it never saw, each as a grid of 8 x 8 squares.
digits_new, labels_new, digit_network = fit_digit_network()
new_grids = digits_new.reshape(-1, 8, 8)
moved_grids = np.zeros_like(new_grids)
moved_grids[:, :, 1:] = new_grids[:, :, :-1]        # every column one pixel to the right
moved_rows = moved_grids.reshape(-1, 64)            # back to 64 numbers a digit
print(f"New digits read right: {digit_network.score(digits_new, labels_new):.0%}")
print(f"The same digits, moved one pixel: {digit_network.score(moved_rows, labels_new):.0%}")

## What the Hidden Layer Learned

- **The hidden layer, “the middle”:** the digits network’s 100 neurons,
  each adding up the 64 squares, then bending: 100 new columns. The last
  layer reads the digit from them.
- **On the slide, four of them,** each with the five new digits that
  turn it on most: two fire for one digit, two for a stroke two digits
  share.
- **Nobody told it about strokes,** and this plain network has no
  detectors: the walk downhill built these features to tell digits
  apart. **Representation learning.**

## One Middle, Many Uses

In [ ]:
# The digits network of "A Picture Is a Grid of Numbers", fitted again. Its middle: the 100
# hidden neurons, each a feature it learned from the 64 squares. New questions get a NEW LAST
# LAYER on the middle (a logit), compared with the same logit on the 64 raw squares. Scored on
# the quarter of the digits the network never saw. About 2 seconds.
digits = load_digits()
digits_fit, digits_new, labels_fit, labels_new = train_test_split(
    digits.data, digits.target, test_size=0.25, random_state=0)
digit_network = MLPClassifier(random_state=0).fit(digits_fit, labels_fit)


def middle_of(grids):
    """The middle: each hidden neuron's value (add, then bend) for each digit."""
    return np.maximum(0, grids @ digit_network.coefs_[0] + digit_network.intercepts_[0])


new_questions = {"Even?": lambda labels: labels % 2 == 0,
                 "5 or more?": lambda labels: labels >= 5,
                 "0, 6, 8 or 9?": lambda labels: np.isin(labels, [0, 6, 8, 9])}
read_right = {}
for question, answer in new_questions.items():
    on_middle = LogisticRegression(max_iter=5000).fit(middle_of(digits_fit), answer(labels_fit))
    on_squares = LogisticRegression(max_iter=5000).fit(digits_fit, answer(labels_fit))
    read_right[question] = (100 * on_middle.score(middle_of(digits_new), answer(labels_new)),
                            100 * on_squares.score(digits_new, answer(labels_new)))
    print(f"{question:14s} new last layer on the middle {read_right[question][0]:.0f}%, "
          f"logit on the squares {read_right[question][1]:.0f}%")

- **The middle of the last slide:** features the network learned to read
  digits. A new question needs only a **new last layer** on it: a logit.
- **Even? 5 or more? 0, 6, 8 or 9?** The middle answers each better than
  the 64 raw squares.
- **One middle, many outputs:** a car’s camera network finds lanes, cars
  and signs, each with its own last layer.

## Pre-Train, Fine-Tune

|  | **1. Pre-train** on lots of data | **2. Fine-tune** on your few rows |
|:---------------|:---------------------------|:---------------------------|
| Language | The next word piece, 300 billion read | 13,000 answers people wrote, and rankings |
| Pictures | ImageNet: 1.2 million photos | A few thousand medical scans |
| The digits | The digits 0 to 4 | A few examples of 5 to 9 |

- **Both steps are training,** the walk downhill that sets the weights.
  Fine-tuning starts from the pre-trained weights, not random ones, so a
  few rows suffice.
- **Not the tuning of the dials:** tuning picks settings, such as λ or
  the number of neurons, on validation data; fine-tuning moves the
  weights themselves.
- **Fine-tuned beats big:** people preferred a fine-tuned model with 1.3
  billion weights to GPT-3, with 175 billion and no fine-tuning (Ouyang
  et al. 2022; ChatGPT was trained the same way).

## Pre-Train, Fine-Tune: The Digits

- **Pre-train** a network on the digits 0 to 4; keep its middle.
  **Fine-tune** for 5 to 9: fit only a new last layer, on a few examples
  of each.
- **With 5 examples of each,** the pre-trained middle reads clearly more
  of the new 5s to 9s than a network from scratch. With 50 each, the gap
  closes.
- **Fair warning:** these tiny pictures are easy; a plain logit on the
  squares does as well. Pre-training pays where inputs are hard to read
  and labels are few.

## Past the Edge of the Data

- **Boosted trees go flat:** past the edge of the data, each tree
  repeats its edge leaf.
- **The network keeps going** in a straight line, near its slope at the
  edge: close to the trend, far below the curve once the curve turns.
- **Neither knows the world past the data:** on the trend the network is
  closer, on the curve the trees. In finance: a month wilder than any in
  the fitting years.

**Try it.** The straight trend of the left panel: boosted trees and a
network, each fitted to its 300 points, then asked about x at the edge
of the data and past it. Then try 3 or −2.

In [ ]:
# 300 noisy points of a straight trend, y = x, with x between -1 and 1.
trend_draw = np.random.default_rng(7)
x_trend = np.sort(trend_draw.uniform(-1, 1, 300))
y_trend = x_trend + trend_draw.normal(0, 0.3, 300)
X_trend = x_trend.reshape(-1, 1)
trend_trees = GradientBoostingRegressor(random_state=0).fit(X_trend, y_trend)
trend_network = MLPRegressor(hidden_layer_sizes=(100,), learning_rate_init=0.01, max_iter=5000,
                             random_state=0).fit(X_trend, y_trend)
for x_new in [1.0, 1.5, 2.0]:                       # the data end just below 1
    trees_say = trend_trees.predict([[x_new]])[0]
    network_says = trend_network.predict([[x_new]])[0]
    print(f"x = {x_new}: boosted trees {trees_say:.2f}, network {network_says:.2f}, "
          f"truth {x_new:.2f}")

# Close

## The Rest, Briefly

- **Convolutional networks,** the detectors of the digits, also read
  price charts as images (Jiang, Kelly & Xiu 2023).
- **Autoencoders** squeeze many inputs through a narrow layer: many
  returns into a few factors (Gu, Kelly & Xiu 2021).
- **Embeddings** turn words into numbers, so that similar words sit
  close together. More with text.
- **Large language models:** transformers trained to predict the next
  word, on vast amounts of text. More with text.

## Readings

**Required**

- Gu, Kelly & Xiu (2020), “Empirical Asset Pricing via Machine
  Learning”, *Review of Financial Studies* 33(5), 2223–2273: the
  introduction and Tables 1, 6 and 7.
- James, Witten, Hastie, Tibshirani & Taylor (2023), *ISLP*, sections
  10.1–10.3 and 10.5–10.7 (networks, convolution, memory, when to use
  them, fitting).

**Optional**

- Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*,
  sections 11.3–11.5.
- Prince (2023), *Understanding Deep Learning*, chapters 3, 6, 7 and 10;
  free at udlbook.github.io.
- Fischer & Krauss (2018), “Deep Learning with Long Short-Term Memory
  Networks for Financial Market Predictions”, *European Journal of
  Operational Research* 270(2), 654–669.